# Classical aspect-based sentiment analysis

## Problem and result at a glance

This case study refactors my University of Bath NLP coursework into a small, tested classical ABSA system. The question is practical: can a transparent sequence-labelling and rule-based pipeline turn annotated product reviews into aspect-level sentiment records and product summaries?

The notebook is organised as a worked data flow. Each stage shows an intermediate object, states the design choice that produced it, and links to the implementation. The current seed-42 results are read from the checked-in aggregate artifacts; the original Hu--Liu corpus is used locally for the preprocessing example.

**Source map:** [data boundaries](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/data.py), [aspect extraction](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/aspects.py), [opinion induction](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/opinions.py), [linking](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/linking.py), [evaluation](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/evaluation.py), [summaries and publication](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/summaries.py) / [experiments](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/experiments.py).

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
import sys; sys.path.insert(0, str(ROOT / "src"))

In [2]:
manifest = json.loads((ARTIFACTS / "run_manifest.json").read_text())
corpus = pd.read_csv(ARTIFACTS / "corpus_summary.csv")
baselines = pd.read_csv(ARTIFACTS / "ate_baselines.csv")
crf = pd.read_csv(ARTIFACTS / "crf_metrics.csv")
negation = pd.read_csv(ARTIFACTS / "negation_ablation.csv")
linking = pd.read_csv(ARTIFACTS / "linking_metrics.csv")
end_to_end = pd.read_csv(ARTIFACTS / "end_to_end_metrics.csv")
summaries = pd.read_csv(ARTIFACTS / "product_summaries.csv")

In [3]:
best = baselines.query("match == 'exact'").sort_values("f1").iloc[-1]
crf_dev = crf.query("split == 'dev' and match == 'exact'").iloc[0]
selected = manifest["selected_linker"]
link_dev = linking.query("method == @selected").iloc[0]
e2e = end_to_end.query("match == 'exact'").iloc[0]
display(Markdown(f"Best baseline exact F1: **{best.f1:.3f}**; CRF dev exact F1: **{crf_dev.f1:.3f}**; selected `{selected}` linker macro-F1/coverage: **{link_dev.macro_f1:.3f}/{link_dev.coverage:.3f}**; test exact F1: **{e2e.f1:.3f}**."))

Best baseline exact F1: **0.039**; CRF dev exact F1: **0.291**; selected `pmi` linker macro-F1/coverage: **0.660/0.415**; test exact F1: **0.167**.

## System architecture

The architecture keeps the useful vertical logic of the original coursework, but makes the boundaries explicit: parse and mask first; fit all learned state on training reviews; select the linker on development data; then run the frozen pipeline once on test data. The result is a staged system whose intermediate outputs can be inspected rather than hidden inside one notebook cell.

![Corrected classical ABSA architecture](../assets/pipeline_architecture.svg)

**Implementation:** [`run_experiment`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/experiments.py) and the [shared domain objects](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/schema.py).

## Data and evaluation boundary

The three related Hu--Liu corpora contain 3,267 parsed reviews, 17 products or domains, and 5,775 non-neutral aspect labels. The official [UIC dataset page](https://www.cs.uic.edu/~liub/FBS/sentiment-analysis.html) identifies the five-product collection with Hu and Liu's KDD-2004 work and the additional nine-product collection with later opinion-mining research. The primary feature-summarisation paper is [Hu and Liu (2004)](https://www.cs.uic.edu/~liub/publications/kdd04-revSummary.pdf).

The input is sentence-scoped because BIO labels, dependency paths, linking, and evaluation all need a common unit. Reviews—not sentences—are split 60/20/20 so near-duplicate text cannot cross the train, development, and test boundaries. Inline supervision is masked with spaces before spaCy processing; this prevents the CRF from reading its answer while preserving source offsets.

Gold aspects are used to train and diagnose the labelled components. They are not used to create deployable links or product summaries.

In [4]:
import spacy
from review_absa.data import _aspect_pairs, clean_text_length_preserving

SAMPLE_PATH = ROOT / "data/raw/Reviews-9-products/ipod.txt"
FALLBACK_SAMPLE = "Battery life[+1]## Battery life keeps improving on the iPod, and a new iPod will give you up to twelve hours of play time, per charge."
lines = SAMPLE_PATH.read_text(encoding="utf-8").splitlines() if SAMPLE_PATH.exists() else [FALLBACK_SAMPLE]
sample_text = next(line.strip() for line in lines if "Battery life[+1]##" in line)

## Raw corpus example

This is a short real example from the published Hu--Liu review collection. The annotation prefix is not ordinary review prose: `Battery life[+1]` is supervision, `##` separates it from the review sentence, and the sentence after the separator is the model input.

The example exercises [`_aspect_pairs`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/data.py) and [`clean_text_length_preserving`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/data.py).

In [5]:
pairs = _aspect_pairs(sample_text)
cleaned = clean_text_length_preserving(sample_text)
review_text = sample_text.split("##", 1)[1].strip()
prefix_end = sample_text.index("##") + 2
visible_clean = cleaned[:prefix_end].replace(" ", "·") + cleaned[prefix_end:]
display(pd.DataFrame([{"stage": "raw annotated", "text": sample_text}, {"stage": "clean model input", "text": visible_clean}, {"stage": "parsed supervision", "text": str(pairs)}]))

,stage,text
0,raw annotated,Battery life[+1]## Battery life keeps improvin...
1,clean model input,·················· Battery life keeps improvin...
2,parsed supervision,"(('battery life', 1, 0),)"


In [6]:
nlp = spacy.load("en_core_web_sm")
sample_doc = nlp(review_text)
gold_words = pairs[0][0].split()
token_rows = [{"token": token.text, "lemma": token.lemma_, "pos": token.pos_, "gold_BIO": "B" if index == 0 else "I" if index < len(gold_words) else "O"} for index, token in enumerate(sample_doc)]
pd.DataFrame(token_rows)

,token,lemma,pos,gold_BIO
0,Battery,battery,NOUN,B
1,life,life,NOUN,I
2,keeps,keep,VERB,O
3,improving,improve,VERB,O
4,on,on,ADP,O
5,the,the,DET,O
6,iPod,iPod,PROPN,O
7,",",",",PUNCT,O
8,and,and,CCONJ,O
9,a,a,DET,O


The transformation makes the central preprocessing trade-off visible: the model receives the natural sentence, but the labelled prefix is replaced by same-length whitespace. The first two tokens therefore receive `B I` supervision in this worked example. The implementation records unalignable annotations instead of silently inventing spans.

**Why this choice:** character-preserving masking protects against label leakage without invalidating offsets. **Consequence:** some historical annotations cannot be projected exactly onto clean token spans, so the alignment rate is reported rather than hidden.

In [7]:
alignment = manifest["training_alignment"]
display(Markdown(f"Training BIO alignment: **{alignment['aligned_aspects']:,}/{alignment['total_gold_aspects']:,}** aspects ({alignment['match_rate']:.1%}). Split sizes: `{manifest['split_review_counts']}`."))

Training BIO alignment: **2,547/3,414** aspects (74.6%). Split sizes: `{'dev': 653, 'test': 653, 'train': 1961}`.

## Aspect extraction

The first question is span discovery. Noun-chunk and frequency pipelines are useful baselines because they are interpretable and high-recall, but they apply fixed rules. The final extractor is a BIO-tagged linear-chain CRF with lexical, POS, dependency, shape, affix, and local-context features; the feature construction is in [`_token_features`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/aspects.py).

A CRF scores a complete tag sequence rather than classifying tokens independently:

$$
P(\mathbf{y}\mid\mathbf{x}) = \frac{\exp\left(\sum_i \mathbf{w}^{\mathsf T}\mathbf{f}(y_{i-1},y_i,\mathbf{x},i)\right)}{Z(\mathbf{x})}.
$$

This is the rationale for using a structured model: BIO transitions can learn coherent span boundaries. The formulation follows [Lafferty, McCallum and Pereira (2001)](https://www.cs.cmu.edu/~epxing/Class/10708-19/notes/10708_scribe_notes.pdf), while BIO chunking follows [Ramshaw and Marcus (1995)](https://aclanthology.org/W95-0107/).

In [8]:
baseline_exact = baselines.query("match == 'exact'")[['pipeline', 'precision', 'recall', 'f1']]
crf_exact = crf.query("split == 'dev' and match == 'exact'").assign(pipeline="crf")[['pipeline', 'precision', 'recall', 'f1']]
pd.concat([baseline_exact, crf_exact], ignore_index=True).sort_values("f1", ascending=False)

,pipeline,precision,recall,f1
5,crf,0.477801,0.209066,0.290862
4,statistical_then_linguistic,0.036885,0.041628,0.039113
3,linguistic_then_statistical,0.036349,0.041628,0.038810
1,linguistic,0.022847,0.111933,0.037949
0,raw,0.017100,0.111933,0.029668
2,statistical,0.015935,0.041628,0.023047


![Aspect extraction comparison](../artifacts/figures/ate_comparison.png)

**What the plot shows:** the CRF is more selective than the raw noun-phrase candidates. Exact matching tests complete surface spans; head matching asks whether the final aspect head was recovered. The gap is therefore evidence about boundary errors, not just a second arbitrary score.

**Implementation:** [`evaluate_aspects`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/evaluation.py) and [`CRFAspectExtractor`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/aspects.py).

## Opinion induction and negation

The corpus labels aspect polarity but does not label opinion spans. I therefore induce a domain-specific lexicon from adjective and verb lemmas near aligned positive and negative training aspects, rather than importing a general-purpose sentiment list. This keeps the source of polarity evidence visible and respects the training boundary; see [`OpinionLexicon`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/opinions.py).

For lemma $w$, the smoothed polarity score is a log-odds difference:

$$
s(w)=\log \frac{C_+(w)+\alpha}{N_+ + \alpha V} - \log \frac{C_-(w)+\alpha}{N_- + \alpha V}.
$$

The choice is related to association-based sentiment induction in [Church and Hanks (1990)](https://aclanthology.org/J90-1003/) and the feature/opinion framing in [Liu (2012)](https://doi.org/10.1561/1500000011). At inference, a small left-context negation rule flips the sign for constructions such as “not good”.

In [9]:
negation[['method', 'negation', 'coverage', 'accuracy', 'macro_f1']].sort_values(['method', 'negation'])

,method,negation,coverage,accuracy,macro_f1
4,dep,disabled,0.649399,0.720798,0.560448
0,dep,enabled,0.649399,0.740741,0.628386
6,dep->pmi,disabled,0.655874,0.720733,0.559434
2,dep->pmi,enabled,0.655874,0.740480,0.627055
5,pmi,disabled,0.415356,0.779510,0.588283
1,pmi,enabled,0.415356,0.795100,0.660050
7,pmi->dep,disabled,0.655874,0.724965,0.564936
3,pmi->dep,enabled,0.655874,0.755994,0.650132


## Aspect-opinion linking

Linking decides which opinion candidate belongs to which aspect. Dependency paths are interpretable but can miss long or poorly parsed relations; PMI can recover recurring associations but is vulnerable to sparse or accidental co-occurrence. The four strategies and their tie-breaking rules are implemented in [`link_aspects`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/linking.py).

For aspect head $a$ and opinion lemma $o$, the association score is:

$$
\mathrm{PMI}(a,o)=\log \frac{P(a,o)}{P(a)P(o)} \approx \log \frac{C(a,o)S}{C(a)C(o)},
$$

where $S$ is the number of training sentences. The development rule selects macro-F1 first and coverage second: it makes the precision/coverage trade-off explicit instead of choosing a method because it looks plausible. The dependency motivation follows [Qiu et al. (2011)](https://aclanthology.org/J11-1002/). Gold aspects appear here only to isolate the linker diagnostic; deployment receives CRF-predicted aspects.

In [10]:
linking.sort_values(["macro_f1", "coverage"], ascending=False)

,split,method,coverage,accuracy,macro_f1,n_total,n_covered
1,dev,pmi,0.415356,0.795100,0.660050,1081,449
3,dev,pmi->dep,0.655874,0.755994,0.650132,1081,709
0,dev,dep,0.649399,0.740741,0.628386,1081,702
2,dev,dep->pmi,0.655874,0.740480,0.627055,1081,709


![Linker accuracy and coverage](../artifacts/figures/linking_tradeoff.png)

PMI wins the declared development rule by accepting lower coverage for more reliable polarity among covered gold aspects. That is useful as a component diagnosis, but it is not a deployment result.

**Implementation:** [`evaluate_linking_component`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/evaluation.py), which retains structured internal error records while publishing aggregate CSV columns only.

## End-to-end inference

The deployable path is deliberately separate: the CRF predicts aspects, the frozen lexicon proposes opinions, and the selected linker creates aspect--opinion--polarity records. Gold aspect text is not supplied to this path.

For the predicted and gold pair sets, the aggregate metrics are:

$$
\mathrm{precision}=\frac{TP}{TP+FP},\qquad
\mathrm{recall}=\frac{TP}{TP+FN},\qquad
F_1=\frac{2PR}{P+R}.
$$

Link coverage is the fraction of predicted aspects receiving a non-neutral link. Exact and head matching follow the complementary boundary/semantic view used in [Pontiki et al. (2014)](https://aclanthology.org/S14-2004/).

**Implementation:** [`evaluate_aspect_polarity`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/evaluation.py).

In [11]:
end_to_end

,split,linker,match,precision,recall,f1,true_positive,false_positive,false_negative,predicted,gold,linked_aspect_coverage
0,test,pmi,exact,0.394737,0.105717,0.166770,135,207,1142,342,1277,0.611993
1,test,pmi,head,0.414706,0.110588,0.174613,141,199,1134,340,1275,0.611993


## Product summaries and error analysis

The summary is the user-facing consequence of the design: it aggregates only links created from **CRF-predicted aspects**. Gold aspects are reserved for training and the labelled linker diagnostic. `unique_reviews` prevents one repetitive review from dominating a product feature, while `occurrences` preserves the raw event count.

**Implementation:** [`make_product_summary`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/summaries.py) and the publication path in [`experiments.py`](https://github.com/boredjnurseman/classical-absa-sentiment-analysis/blob/main/src/review_absa/experiments.py).

In [12]:
summaries.query("count_mode == 'unique_reviews'").sort_values(['total', 'product', 'aspect'], ascending=[False, True, True]).head(15)

,product,aspect,positive,negative,total,count_mode
191,Speaker,speakers,10,1,11,unique_reviews
71,Creative Labs Nomad Jukebox Zen Xtra 40GB,software,7,3,10,unique_reviews
192,Speaker,sound,8,0,8,unique_reviews
72,Creative Labs Nomad Jukebox Zen Xtra 40GB,player,2,4,6,unique_reviews
58,Computer,picture,3,1,4,unique_reviews
73,Creative Labs Nomad Jukebox Zen Xtra 40GB,interface,4,0,4,unique_reviews
74,Creative Labs Nomad Jukebox Zen Xtra 40GB,size,4,0,4,unique_reviews
75,Creative Labs Nomad Jukebox Zen Xtra 40GB,sound,4,0,4,unique_reviews
98,Diaper Champ,diaper champ,3,1,4,unique_reviews
122,Linksys Router,router,3,1,4,unique_reviews


![Predicted product summaries](../artifacts/figures/product_summaries.png)

The aggregate failure chain is visible: unalignable annotations reduce available BIO supervision; missed or fragmented aspects cannot be linked; conservative PMI leaves some surviving aspects uncovered. The `AlignmentReport` keeps the locations and reasons internally, while the public manifest exposes only aggregate reason counts. The absence of gold opinion spans means triplet-level ASTE claims would overstate what this corpus can support.

## Conclusions

The value of this system is not state-of-the-art performance. It is a defensible classical pipeline whose choices are inspectable: structural corpus parsing, leakage-safe masking, review-level splits, train-fitted lexical/statistical state, structured BIO extraction, explicit linker selection, predicted-aspect summaries, and transactional aggregate publication.

The results show both the usefulness and the limits of that design. The CRF improves selective span extraction over heuristic baselines; PMI gives the strongest covered-case linker score; and the lower joint recall exposes the cost of composing imperfect stages. That is the engineering lesson: intermediate evidence and explicit boundaries make failure measurable rather than invisible.

## References

- [Hu, M. and Liu, B. (2004). *Mining and summarizing customer reviews*. KDD-2004.](https://www.cs.uic.edu/~liub/publications/kdd04-revSummary.pdf)
- [Hu, M. and Liu, B. (2004). *Mining opinion features in customer reviews*. AAAI-04.](https://www.cs.uic.edu/~liub/publications/aaai04-featureExtract.pdf)
- [Lafferty, J., McCallum, A. and Pereira, F. (2001). *Conditional random fields*.](https://www.cs.cmu.edu/~epxing/Class/10708-19/notes/10708_scribe_notes.pdf)
- [Ramshaw, L. and Marcus, M. (1995). *Text chunking using transformation-based learning*.](https://aclanthology.org/W95-0107/)
- [Church, K. and Hanks, P. (1990). *Word association norms, mutual information, and lexicography*.](https://aclanthology.org/J90-1003/)
- [Qiu, G. et al. (2011). *Opinion word expansion and target extraction through double propagation*.](https://aclanthology.org/J11-1002/)
- [Pontiki, M. et al. (2014). *SemEval-2014 Task 4: Aspect Based Sentiment Analysis*.](https://aclanthology.org/S14-2004/)
- [Liu, B. (2012). *Sentiment Analysis and Opinion Mining*.](https://doi.org/10.1561/1500000011)
- [Official Hu--Liu dataset page (UIC).](https://www.cs.uic.edu/~liub/FBS/sentiment-analysis.html)